In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import make_column_transformer

Preliminary:

- Load the [breast-cancer.csv](./data/breast-cancer.csv) file
- Drop `Class` column
- Drop NaN values
- Split the data in a train set and test set (test set size = 20% of the total size) with `random_state=43`.

In [2]:
columns =[
        'age',
        'menopause',
        'tumor-size',
        'inv-nodes',
        'node-caps',
        'deg-malig',
        'breast',
        'breast-quad',
        'irradiat',
        'class'
    ]
df = pd.read_csv('./data/breast-cancer.csv', sep=',', names=columns, na_values="?")
df.head()

,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat,class
0,40-49,premeno,15-19,0-2,yes,3,right,left_up,no,recurrence-events
1,50-59,ge40,15-19,0-2,no,1,right,central,no,no-recurrence-events
2,50-59,ge40,35-39,0-2,no,2,left,left_low,no,recurrence-events
3,40-49,premeno,35-39,0-2,yes,3,right,left_low,yes,no-recurrence-events
4,40-49,premeno,30-34,3-5,yes,2,left,right_up,no,recurrence-events


In [3]:
print(df.shape)
df = df.dropna()
print(df.shape)

X_df = df.drop(columns=['class'])
y = df['class']

(286, 10)
(277, 10)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2,random_state=43)
print(X_train)

       age menopause tumor-size inv-nodes node-caps  deg-malig breast  \
210  40-49      ge40      25-29     12-14       yes          3   left   
111  60-69      ge40      30-34       0-2        no          2   left   
17   60-69      ge40      15-19       0-2        no          2  right   
170  50-59      ge40      20-24       0-2        no          2  right   
146  30-39   premeno      30-34      9-11        no          2  right   
..     ...       ...        ...       ...       ...        ...    ...   
194  40-49   premeno      25-29       0-2        no          2  right   
152  50-59      ge40      40-44       3-5       yes          2   left   
16   50-59   premeno      10-14       3-5        no          1  right   
62   40-49      ge40      20-24       0-2        no          3   left   
263  40-49   premeno      20-24       3-5       yes          2  right   

    breast-quad irradiat  
210   right_low      yes  
111    left_low      yes  
17      left_up       no  
170     central

1. Count the number of unique values per feature in the train set.

In [5]:
for col in X_train.columns:
    print(f"number of unique values of column {col}: {X_train[col].nunique()}")

number of unique values of column age: 6
number of unique values of column menopause: 3
number of unique values of column tumor-size: 11
number of unique values of column inv-nodes: 6
number of unique values of column node-caps: 2
number of unique values of column deg-malig: 3
number of unique values of column breast: 2
number of unique values of column breast-quad: 5
number of unique values of column irradiat: 2


2. Identify the ordinal variables, nominal variables, and the target. Compute a OneHotEncoder transformation on the test set for all categorical features (no ordinal) in the following order `['node-caps' , 'breast', 'breast-quad', 'irradiat']`. Here are the assumptions made on the variables:

```console
age: Ordinal
['90-99' > '80-89' > '70-79' > '60-69' > '50-59' > '40-49' > '30-39' > '20-29' > '10-19']

menopause: Ordinal
['ge40'> 'premeno' >'lt40']

tumor-size: Ordinal
['55-59' > '50-54' > '45-49' > '40-44' > '35-39' > '30-34' > '25-29' > '20-24' > '15-19' > '10-14' > '5-9' > '0-4']

inv-nodes: Ordinal
['36-39' > '33-35' > '30-32' > '27-29' > '24-26' > '21-23' > '18-20' > '15-17' > '12-14' > '9-11' > '6-8' > '3-5' > '0-2']

node-caps: One Hot
['yes' 'no']

deg-malig: Ordinal
[3 > 2 > 1]

breast: One Hot
['left' 'right']

breast-quad: One Hot
['right_low' 'left_low' 'left_up' 'central' 'right_up']

irradiat: One Hot
['yes' 'no']

Class: Target (One Hot)
['recurrence-events' 'no-recurrence-events']
```

- Fit on the train set

- Transform the test set

Example of expected output:

```console
# OneHotEncoder on: ['node-caps' , 'breast', 'breast-quad', 'irradiat']

input: ohe.transform(X_test[ohe_cols])[:10]
output:
array([[1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0.],
       [1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 1.],
       [1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0.],
       [1., 0., 1., 0., 0., 0., 0., 1., 0., 1., 0.],
       [1., 0., 0., 1., 0., 0., 0., 0., 1., 1., 0.],
       [1., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0.],
       [1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1.],
       [1., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0.]])

input: ohe.get_feature_names_out(ohe_cols)
output:
array(['node-caps_no', 'node-caps_yes', 'breast_left', 'breast_right',
       'breast-quad_central', 'breast-quad_left_low',
       'breast-quad_left_up', 'breast-quad_right_low',
       'breast-quad_right_up', 'irradiat_no', 'irradiat_yes'],
      dtype=object)

```

In [14]:
# node_caps=['yes' 'no']
# breast=['left' 'right']
# breast_quad=['right_low' 'left_low' 'left_up' 'central' 'right_up']
# irradiat=['yes' 'no']
# Class=['recurrence-events' 'no-recurrence-events']
one_hot_encode_cols =['node-caps', 'breast', 'breast-quad', 'irradiat']


one_hot_encode = OneHotEncoder(sparse_output=False)

one_hot_encode.fit(X_train[one_hot_encode_cols])
X_test_one_hot_encode = one_hot_encode.transform(X_test[one_hot_encode_cols])

print(X_test_one_hot_encode[:10])

print(one_hot_encode.get_feature_names_out(one_hot_encode_cols))


[[1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0.]
 [1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 0.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]
 [1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1.]
 [1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]]
['node-caps_no' 'node-caps_yes' 'breast_left' 'breast_right'
 'breast-quad_central' 'breast-quad_left_low' 'breast-quad_left_up'
 'breast-quad_right_low' 'breast-quad_right_up' 'irradiat_no'
 'irradiat_yes']


3. Create one Ordinal encoder for all Ordinal features in the following order `["menopause", "age", "tumor-size","inv-nodes", "deg-malig"]` on the test set. The documentation of Scikit-learn is not clear on how to perform this on many columns at the same time. Here's a **hint**:

If the ordinal dataset is (subset of two columns, but I keep all rows for this example):

    |    | menopause     |   deg-malig |
    |---:|:--------------|------------:|
    |  0 | premeno       |           3 |
    |  1 | ge40          |           1 |
    |  2 | ge40          |           2 |
    |  3 | premeno       |           3 |
    |  4 | premeno       |           2 |

The first step is to create a dictionary or a list - the most recent versions of sklearn take lists as input:

```console
dict_ = {0: ['lt40', 'premeno' , 'ge40'], 1:[1,2,3]}
```

Then to instantiate an `OrdinalEncoder`:

```console
oe = OrdinalEncoder(dict_)
```

Now that you have enough information:

- Fit on the train set
- Transform the test set

In [15]:
ord_cols = ["menopause", "age", "tumor-size", "inv-nodes", "deg-malig"]

menopause_cats = ['lt40', 'premeno', 'ge40']
age_cats = ['10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']
tumor_size_cats = ['0-4', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59']
inv_nodes_cats = ['0-2', '3-5', '6-8', '9-11', '12-14', '15-17', '18-20', '21-23', '24-26', '27-29', '30-32', '33-35', '36-39']
deg_malig_cats = [1, 2, 3]

categories_list = [menopause_cats, age_cats, tumor_size_cats, inv_nodes_cats, deg_malig_cats]

ordinal_encoder = OrdinalEncoder(categories=categories_list)

ordinal_encoder.fit(X_train[ord_cols])
X_train_ordinal_encoder = ordinal_encoder.transform(X_train[ord_cols])
X_test_ordinal_encoder = ordinal_encoder.transform(X_test[ord_cols])

print(X_test_ordinal_encoder)

print(ordinal_encoder.get_feature_names_out(ord_cols))

[[ 2.  5.  2.  0.  1.]
 [ 2.  5.  2.  0.  0.]
 [ 2.  5.  4.  5.  2.]
 [ 1.  4.  5.  1.  1.]
 [ 2.  5.  5.  0.  2.]
 [ 1.  2.  1.  0.  1.]
 [ 1.  2.  8.  0.  1.]
 [ 2.  5.  2.  0.  0.]
 [ 2.  5.  5.  0.  2.]
 [ 1.  2.  3.  0.  0.]
 [ 1.  2.  5.  2.  2.]
 [ 2.  4.  5.  0.  0.]
 [ 1.  3.  5.  0.  0.]
 [ 1.  3.  7.  0.  1.]
 [ 1.  3.  6.  1.  1.]
 [ 2.  5.  6.  0.  2.]
 [ 1.  4.  7.  5.  2.]
 [ 2.  4.  6.  2.  1.]
 [ 1.  2.  8.  0.  1.]
 [ 1.  2.  0.  0.  1.]
 [ 1.  4. 10.  3.  1.]
 [ 2.  4.  0.  0.  1.]
 [ 1.  2.  7.  0.  2.]
 [ 2.  4.  4.  0.  1.]
 [ 2.  4.  7.  5.  2.]
 [ 2.  6.  2.  0.  1.]
 [ 1.  2.  4.  0.  2.]
 [ 1.  4.  5.  0.  1.]
 [ 0.  4.  6.  0.  2.]
 [ 1.  4.  5.  0.  1.]
 [ 2.  4.  2.  0.  1.]
 [ 2.  5.  4.  1.  1.]
 [ 2.  4.  0.  0.  0.]
 [ 1.  3.  2.  0.  1.]
 [ 1.  3.  5.  0.  1.]
 [ 2.  5.  3.  0.  1.]
 [ 1.  4.  6.  0.  0.]
 [ 2.  5.  9.  2.  2.]
 [ 2.  3.  5.  0.  1.]
 [ 1.  2.  5.  0.  1.]
 [ 1.  3.  5.  0.  1.]
 [ 1.  3.  7.  0.  2.]
 [ 2.  5.  3.  0.  1.]
 [ 2.  5.  

4. Use a `make_column_transformer` to combine the two Encoders.

- Fit on the train set
- Transform the test set

_Hint: Check the first resource_

**Note: The version 0.22 of Scikit-learn can't handle `get_feature_names` on `OrdinalEncoder`. If the column transformer contains an `OrdinalEncoder`, the method returns this error**:

```console
AttributeError: Transformer ordinalencoder (type OrdinalEncoder) does not provide get_feature_names.
```

**It means that if you want to use the Ordinal Encoder, you will have to create a variable that contains the column names in the correct order. This step is not required in that exercise**

Resources:

- [Resource 2](https://towardsdatascience.com/guide-to-encoding-categorical-features-using-scikit-learn-for-machine-learning-5048997a5c79)


In [16]:
ct = make_column_transformer(
    (OneHotEncoder(sparse_output=False), one_hot_encode_cols),
    (OrdinalEncoder(categories=categories_list), ord_cols),
    remainder='drop'
)


ct.fit(X_train)
X_test_transformed = ct.transform(X_test)

print("4. Column Transformer Combined Output (first 5 rows):\n")
print(X_test_transformed[:5])

4. Column Transformer Combined Output (first 5 rows):

[[1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 2. 5. 2. 0. 1.]
 [1. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 2. 5. 2. 0. 0.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 2. 5. 4. 5. 2.]
 [0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 4. 5. 1. 1.]
 [1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 0. 2. 5. 5. 0. 2.]]
